# 🏎 KEIBA-AI 日曜結果 v7

**日曜18時以降に実行**

## v7からの変更点
- ✅ fetch_results を jra_scraper.py から呼び出すよう変更（regex バグ解消）
- ✅ save_history_db / check_and_update_bets をモジュールから呼び出し
- ✅ セル2に不足ファイル追加（calendar.py, jst.py, bias.py, speed_index.py 等）
- ✅ キャッシュバスティング（?t={timestamp}）追加

| セル | 内容 |
|---|---|
| 1 | セットアップ |
| 2 | src/ モジュール取得 |
| 3 | 初期化 |
| 4 | ★ 結果取得＋保存＋照合 |
| 5 | ROI多次元分析 |

In [ ]:
!pip install -q requests beautifulsoup4 lxml pandas
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, time, sqlite3, requests, subprocess
import pandas as pd
from datetime import datetime, timezone, timedelta
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_DIR = '/content/drive/MyDrive/keiba_ai'
for d in [BASE_DIR, f'{BASE_DIR}/data', f'{BASE_DIR}/app', f'{BASE_DIR}/logs']:
    os.makedirs(d, exist_ok=True)

JST = timezone(timedelta(hours=9))
jst_now = datetime.now(JST)
print(f'✅ セットアップ完了 ({jst_now.strftime("%Y-%m-%d %H:%M:%S")} JST)')

In [ ]:
import urllib.request, time as _time

BASE_URL = 'https://raw.githubusercontent.com/hanagenuku/keiba_ai/main'
SRC_FILES = [
    'src/__init__.py',
    'src/tools/__init__.py',
    'src/tools/tune_weights.py',
    'src/tools/calibrate.py',
    'src/tools/analyze_divergence.py',
    'src/tools/rescrape_history.py',
    'src/tools/bias.py',
    'src/features/__init__.py',
    'src/features/engine.py',
    'src/features/speed_index.py',
    'src/utils/__init__.py',
    'src/utils/config.py',
    'src/utils/db.py',
    'src/utils/model_registry.py',
    'src/utils/jst.py',
    'src/scraper/__init__.py',
    'src/scraper/parser.py',
    'src/scraper/jra_scraper.py',
    'src/scraper/calendar.py',
    'src/models/__init__.py',
    'src/models/calibration.py',
    'src/models/predict.py',
    'src/betting/__init__.py',
    'src/betting/make_bets.py',
    'src/betting/ev_filter.py',
    'src/betting/app_json.py',
]
ts = int(_time.time())
for rel in SRC_FILES:
    dest = f'{BASE_DIR}/{rel}'
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    urllib.request.urlretrieve(f'{BASE_URL}/{rel}?t={ts}', dest)
    print(f'✅ {rel}')
for key in list(sys.modules.keys()):
    if 'src' in key:
        del sys.modules[key]
sys.path.insert(0, BASE_DIR)
print('🔄 アップデート完了')

In [ ]:
# ── セル3: 初期化 ──
BANKROLL = 100_000
FUKU_AMT = 500
TAN_AMT  = 300
WIDE_AMT = 300
REN_AMT  = 300
TAN2_AMT = 300
SAN_AMT  = 300

from src.features.engine import init_engine
from src.utils.db import (init_db, save_history_db, save_results_db, check_and_update_bets)
from src.betting.make_bets import init_betting
from src.scraper.jra_scraper import fetch_results

HEADERS  = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
JRA_BASE = 'https://www.jra.go.jp'
HIST_PATH = f'{BASE_DIR}/data/history.db'
BIAS_PATH = f'{BASE_DIR}/data/track_bias_latest.json'

init_db(BASE_DIR)
init_engine(BASE_DIR)
init_betting(BASE_DIR, bankroll=BANKROLL,
             fuku_amt=FUKU_AMT, tan_amt=TAN_AMT, wide_amt=WIDE_AMT,
             ren_amt=REN_AMT, tan2_amt=TAN2_AMT, san_amt=SAN_AMT)

print('✅ 初期化完了')

In [ ]:
# ── セル4: ★ 結果取得・history.db保存・bet照合 ──

jst_now = datetime.now(JST)
target_date = jst_now.strftime('%Y%m%d')
print(f'📅 本日結果取得: {target_date}')

sess = requests.Session()
sess.headers.update(HEADERS)
retry = Retry(total=3, backoff_factor=2, status_forcelist=[500,502,503,504])
sess.mount('https://', HTTPAdapter(max_retries=retry))
sess.get(f'{JRA_BASE}/keiba/thisweek/', timeout=15)

all_results = fetch_results(sess, target_date)
print(f'📋 取得完了: {len(all_results)}レース')

if all_results:
    surf_counts = {}
    for r in all_results:
        s = r.get('surface', '?')
        surf_counts[s] = surf_counts.get(s, 0) + 1
    print(f'   コース内訳: {surf_counts}')

    save_history_db(all_results, BASE_DIR)
    save_results_db(all_results, BASE_DIR)

    conn = sqlite3.connect(f'{BASE_DIR}/data/history.db')
    dt_max = conn.execute('SELECT MAX(date) FROM race_history').fetchone()[0]
    rc = conn.execute('SELECT COUNT(*) FROM race_history').fetchone()[0]
    hc = conn.execute('SELECT COUNT(*) FROM horse_history').fetchone()[0]
    conn.close()
    print(f'   📊 history.db: {rc:,}レース / {hc:,}出走 / 最新: {dt_max}')

chk = check_and_update_bets(all_results, BASE_DIR)
print(f'\n【照合結果】 {chk["hit"]}/{chk["total"]}的中  '
      f'投資¥{chk["invested"]:,} / 回収¥{chk["recovered"]:,}  '
      f'ROI {chk["roi"]:.1f}%')
for d in chk['details']:
    print(d)

In [ ]:
import sqlite3, pandas as pd

DB_PATH = f'{BASE_DIR}/data/keiba.db'
conn = sqlite3.connect(DB_PATH)
try:
    df_bets = pd.read_sql("""
        SELECT date, race_id, bet_type, horse_num, horse_name,
               odds_est, amount, is_hit, payout,
               ev_rank, racecourse, distance, surface,
               running_style, popularity, ai_score
        FROM bets WHERE is_hit >= 0
    """, conn)
finally:
    conn.close()

if df_bets.empty:
    print('⚠ 集計対象のベットデータがありません')
else:
    df_bets['amount']  = pd.to_numeric(df_bets['amount'],  errors='coerce').fillna(0)
    df_bets['payout']  = pd.to_numeric(df_bets['payout'],  errors='coerce').fillna(0)
    df_bets['ev_rank'] = pd.to_numeric(df_bets['ev_rank'], errors='coerce').fillna(99)
    df_bets['popularity'] = pd.to_numeric(df_bets['popularity'], errors='coerce').fillna(99)

    def roi_summary(g):
        inv = g['amount'].sum(); ret = g['payout'].sum()
        hit = g['is_hit'].sum(); n   = len(g)
        roi = ret / inv * 100 if inv > 0 else 0
        return pd.Series({'件数':n,'的中':int(hit),'的中率':f'{hit/n*100:.1f}%',
                          '投資額':int(inv),'回収額':int(ret),'ROI':f'{roi:.1f}%'})

    print('='*60)
    print('【① EV順位別 ROI】（EV1位が最も期待値が高い）')
    df_bets['EV順位'] = df_bets['ev_rank'].apply(
        lambda x: '1位' if x==1 else '2位' if x==2 else '3位' if x==3 else '4位以下')
    print(df_bets.groupby('EV順位').apply(roi_summary, include_groups=False).to_string())

    print(); print('='*60)
    print('【② 賭け種別 ROI】')
    print(df_bets.groupby('bet_type').apply(roi_summary, include_groups=False).to_string())

    print(); print('='*60)
    print('【③ 競馬場別 ROI】')
    if df_bets['racecourse'].str.strip().any():
        print(df_bets.groupby('racecourse').apply(roi_summary, include_groups=False).to_string())
    else:
        print('（データなし）')

    print(); print('='*60)
    print('【④ 距離帯別 ROI】')
    df_bets['距離帯'] = pd.cut(
        pd.to_numeric(df_bets['distance'], errors='coerce').fillna(0),
        bins=[0,1400,1800,2200,9999],
        labels=['~1400m','1401~1800m','1801~2200m','2201m~'])
    print(df_bets.groupby('距離帯', observed=True).apply(roi_summary, include_groups=False).to_string())

    print(); print('='*60)
    print('【⑤ 人気別 ROI】')
    df_bets['人気帯'] = df_bets['popularity'].apply(
        lambda x: '1~3番人気' if x<=3 else '4~6番人気' if x<=6 else '7~9番人気' if x<=9 else '10番人気以上')
    print(df_bets.groupby('人気帯').apply(roi_summary, include_groups=False).to_string())

    print(); print('='*60)
    print('【⑥ 脚質別 ROI】')
    if df_bets['running_style'].str.strip().any():
        print(df_bets.groupby('running_style').apply(roi_summary, include_groups=False).to_string())
    else:
        print('（データなし）')

    print(); print('='*60)
    print('【⑦ 週次ROI推移】')
    df_bets['week'] = pd.to_datetime(df_bets['date']).dt.to_period('W').astype(str)
    wk = df_bets.groupby('week').apply(roi_summary, include_groups=False).reset_index()
    print(wk[['week','件数','的中率','ROI']].to_string(index=False))

    total_inv = df_bets['amount'].sum(); total_ret = df_bets['payout'].sum()
    total_roi = total_ret / total_inv * 100 if total_inv > 0 else 0
    print(f'\n📊 【通算サマリ】')
    print(f'  通算投資額: ¥{int(total_inv):,}  通算回収額: ¥{int(total_ret):,}  通算ROI: {total_roi:.1f}%  ベット数: {len(df_bets)}件')